In [ ]:
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
from anndata import AnnData
import anndata as ad
import pathlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import skimage
import seaborn as sns
import tangram as tg

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

%load_ext autoreload
%autoreload 2
%matplotlib inline

scanpy==1.10.1 anndata==0.10.7 umap==0.5.6 numpy==1.26.4 scipy==1.13.1 pandas==2.2.2 scikit-learn==1.5.0 statsmodels==0.14.2 igraph==0.11.5 pynndescent==0.5.12
squidpy==1.5.0


In [ ]:
mpl.rcParams["figure.figsize"] = (5, 5)

In [ ]:
adata_st = sc.read('../data/tangram_R5779_TMA3-S4_FINAL.h5ad')
adata_st

AnnData object with n_obs × n_vars = 22431 × 990
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.Histone', 'Max.Histone', 'Mean.G', 'Max.G', 'Mean.rRNA_MembraneStain_B2M', 'Max.rRNA_MembraneStain_B2M', 'Mean.GFAP', 'Max.GFAP', 'Mean.DAPI', 'Max.DAPI', 'cell_ID', 'sample', 'orig.ident', 'nCount_Nanostring', 'nFeature_Nanostring', 'Mean.rRNA_CD298_B2M', 'Max.rRNA_CD298_B2M', 'Slide_name', 'Run_name', 'ISH.concentration', 'Beta', 'tissue', 'Run_Slide_name', 'slide_ID_numeric', 'Run_Tissue_name', 'log10totalcounts', 'IFcolor', 'nb_clus', 'leiden_clus', 'id', 'FOV', 'PMCID', 'Replicate', 'ID_R', 'uniform_density', 'rna_count_based_density', 'x', 'y', 'tangram_prediction', 'tangram_score', 'leiden'
    var: 'n_cells', 'sparsity'
    uns: 'PMCID_colors', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'overlap_genes', 'pca', 'rank_genes_groups', 'spatial', 'tangram_prediction_colors', 'training_genes', 'umap'
    obsm: 'X_pca', 'X_uma

In [ ]:
adata_st2 = sc.read('../data/tangram_R5779_TMA2-S6_FINAL.h5ad')
adata_st2

AnnData object with n_obs × n_vars = 4121 × 990
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.Histone', 'Max.Histone', 'Mean.G', 'Max.G', 'Mean.rRNA_MembraneStain_B2M', 'Max.rRNA_MembraneStain_B2M', 'Mean.GFAP', 'Max.GFAP', 'Mean.DAPI', 'Max.DAPI', 'cell_ID', 'sample', 'orig.ident', 'nCount_Nanostring', 'nFeature_Nanostring', 'Mean.rRNA_CD298_B2M', 'Max.rRNA_CD298_B2M', 'Slide_name', 'Run_name', 'ISH.concentration', 'Beta', 'tissue', 'Run_Slide_name', 'slide_ID_numeric', 'Run_Tissue_name', 'log10totalcounts', 'IFcolor', 'nb_clus', 'leiden_clus', 'id', 'FOV', 'PMCID', 'Replicate', 'ID_R', 'uniform_density', 'rna_count_based_density', 'x', 'y', 'tangram_prediction', 'tangram_score', 'leiden'
    var: 'n_cells', 'sparsity'
    uns: 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'overlap_genes', 'pca', 'rank_genes_groups', 'spatial', 'tangram_prediction_colors', 'training_genes', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial', 's

In [ ]:
adata = ad.concat([adata_st, adata_st2], axis=0, merge='same', pairwise=True, index_unique='_')
adata

AnnData object with n_obs × n_vars = 26552 × 990
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.Histone', 'Max.Histone', 'Mean.G', 'Max.G', 'Mean.rRNA_MembraneStain_B2M', 'Max.rRNA_MembraneStain_B2M', 'Mean.GFAP', 'Max.GFAP', 'Mean.DAPI', 'Max.DAPI', 'cell_ID', 'sample', 'orig.ident', 'nCount_Nanostring', 'nFeature_Nanostring', 'Mean.rRNA_CD298_B2M', 'Max.rRNA_CD298_B2M', 'Slide_name', 'Run_name', 'ISH.concentration', 'Beta', 'tissue', 'Run_Slide_name', 'slide_ID_numeric', 'Run_Tissue_name', 'log10totalcounts', 'IFcolor', 'nb_clus', 'leiden_clus', 'id', 'FOV', 'PMCID', 'Replicate', 'ID_R', 'uniform_density', 'rna_count_based_density', 'x', 'y', 'tangram_prediction', 'tangram_score', 'leiden'
    obsm: 'X_pca', 'X_umap', 'spatial', 'spatial_fov', 'tangram_ct_pred'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [ ]:
sc.set_figure_params(scanpy=True, dpi=500, dpi_save=500, frameon=False, vector_friendly=True, fontsize=7, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='png2x')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

probs = adata.obsm['tangram_ct_pred'].copy()
probs.columns = probs.columns.astype(str)

probs['tangram_prediction'] = adata.obs['tangram_prediction'].astype(str)

# Calculate mean probabilities per tangram_prediction
mean_probs = probs.groupby('tangram_prediction').mean()

In [ ]:
new_order = ['RG_like', 'Tri_IPC_like', 'nIPC_like', 'OPC_like', 'COP_like', 'AC_like', 'Cilia_like', 
             'TD_like', 'GW_like', 'Oligodendrocyte', 'Mono_neutrophil', 'TAM_C1Q', 'TAM_lipid_laden', 
             'TAM_scavenger', 'TAM_IL1B', 'TAM_homeostatic', 'T_cell', 'B_cell', 'Endothelial', 'Mural']

colors = [
    '#74150f', '#b26671', '#fdc955', '#c13910', '#f4e5e1', '#caa4ab',
    '#8f7308', '#ba7db3', '#5c2454', '#bae7c2', '#00a69c', '#165884',
    '#7b7dba', '#95bbcd', '#343682', '#26a9e0', '#ffbb78', '#474747',
    '#217236', '#a24600'
]

color_mapping = dict(zip(new_order, colors))

row_colors = pd.Series(new_order, index=new_order).map(color_mapping)
col_colors = row_colors

mean_probs = mean_probs.loc[new_order, new_order]

sns.clustermap(
    mean_probs, 
    cmap='BuPu', 
    annot=False, 
    vmax=0.4, 
    vmin=0.05, 
    linewidths=None, 
    linecolor=None, 
    row_colors=row_colors, 
    col_colors=col_colors,
    col_cluster=False,  
    row_cluster=False 
)

plt.title('Tangram Prediction Scores')
plt.xlabel('Predicted Labels')
plt.savefig('figures/tangram_prediction.pdf')
plt.show()